In [4]:
import json
import pandas as pd
import glob
import os

RAW_MATCH_DIR = "RawData"   # root folder containing year folders
OUT_CSV = "data_processed/match_dates.csv"

rows = []

# Recursively scan all year folders
json_files = glob.glob(
    os.path.join(RAW_MATCH_DIR, "**", "*.json"),
    recursive=True
)

print(f"Found {len(json_files)} JSON files")

for path in json_files:
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        continue  # skip corrupt files safely

    info = data.get("info", {})

    # -----------------------
    # FILTER: T20 ONLY
    # -----------------------
    if info.get("match_type") != "T20":
        continue

    # -----------------------
    # match_id = filename
    # -----------------------
    match_id = os.path.splitext(os.path.basename(path))[0]

    # -----------------------
    # match_date
    # -----------------------
    dates = info.get("dates", [])
    if not dates:
        continue

    match_date = dates[0]

    rows.append({
        "match_id": match_id,
        "match_date": match_date
    })

# -----------------------
# SAFETY CHECK
# -----------------------
if not rows:
    raise RuntimeError("❌ No T20 matches extracted. Check folder structure or JSON format.")

df = pd.DataFrame(rows)

df["match_date"] = pd.to_datetime(df["match_date"], errors="coerce").dt.date

df.to_csv(OUT_CSV, index=False)

print("✅ match_dates.csv generated")
print(f"Total T20 matches extracted: {len(df)}")


Found 12765 JSON files
✅ match_dates.csv generated
Total T20 matches extracted: 8081


merging dates into the venue_match_level

In [5]:
import pandas as pd

vm = pd.read_csv("data_processed/venue_match_level.csv")
dates = pd.read_csv("data_processed/match_dates.csv")

vm["match_id"] = vm["match_id"].astype(str)
dates["match_id"] = dates["match_id"].astype(str)

vm = vm.merge(dates, on="match_id", how="left")

vm.to_csv("data_processed/venue_match_level.csv", index=False)

print("✅ match_date added to venue_match_level.csv")


✅ match_date added to venue_match_level.csv
